# Bloco 3 — Notebook 02: Simulador Causal Single-Slot, Ablação (Degraus A–E) e DSR
Este notebook é **100% autocontido**: se o banco SQLite local ainda não tiver as velas baixadas na VM do Colab, ele mesmo chama `download_and_build_aligned_universe()` antes de rodar a ablação e o **Gate 3**.
- Partição rígida: **60% Treino / 25% Validação / 15% Holdout (6 meses)** com *Purging* de 24h e *Embargo* de 2%.
- Cálculo empírico de $\mathbb{E}[R]$ (incluindo o desfecho real das saídas por barreira de tempo de 24h).
- Ablação explícita do Degrau C com e sem filtro de *Order Flow Delta* (`require_order_flow_filter`).
- Registro automático em `logs/backtest_trials_registry.csv` ($K_{\max} = 20$) e validação do **Gate 3** ($\text{DSR} \ge 0,95$).


In [ ]:
# Bootstrap automático no Google Colab (clona ou atualiza o repositório e recarrega módulos src.*)
import os, subprocess, sys
from pathlib import Path
REPO_URL = 'https://github.com/pac-coins/quant-crypto-chameleon-radar-engine.git'
REPO_DIR = Path('quant-crypto-chameleon-radar-engine')
if 'google.colab' in sys.modules:
    if not Path('config/strategies.yaml').exists():
        if not REPO_DIR.exists():
            subprocess.run(['git', 'clone', REPO_URL], check=True)
        os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pydantic', 'pydantic-settings', 'PyYAML', 'pandas', 'requests', 'cryptography'], check=True)
if '.' not in sys.path:
    sys.path.insert(0, '.')
# Limpa cache de módulos src.* da memória do Python para garantir uso do código atualizado após git pull
for _mod in [k for k in list(sys.modules) if k == 'src' or k.startswith('src.')]:
    del sys.modules[_mod]
os.environ.setdefault('MODE', 'SHADOW_CHAMELEON')
os.environ.setdefault('TELEGRAM_BOT_TOKEN', 'colab_backtest_token_placeholder')
os.environ.setdefault('TELEGRAM_CHAT_ID', '0')


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from src.core.config import load_config
from src.core.database import Database
from src.core.binance_client import MonotonicClock, PublicBinanceClient, WeightGovernor
from src.data.candle_collector import CandleCollector, load_frame, align_btc_context
from src.features.feature_store import FeatureStore, FeatureRequest
from src.regimes.regime_detector import RegimeDetector
from src.strategies.bull_trend_pullback import BullTrendPullbackStrategy

cfg = load_config()
SYMBOLS = list(cfg.strategies.universe.symbols)
START_DATE_STR = cfg.strategies.universe.backtest_min_start_date  # '2022-01-01'
START_MS = int(datetime.fromisoformat(f'{START_DATE_STR}T00:00:00+00:00').timestamp() * 1000)

db_path = Path('data/colab_backtest.sqlite')
db_path.parent.mkdir(parents=True, exist_ok=True)
db = Database(db_path)
db.migrate()  # Cria todas as tabelas (candles, trades, etc.) no SQLite WAL
governor = WeightGovernor()
client = PublicBinanceClient(base_url='https://data-api.binance.vision', governor=governor)
clock = MonotonicClock(safety_margin_ms=cfg.strategies.execution_and_resilience.clock_safety_margin_ms)
detector = RegimeDetector(cfg.strategies.regimes_and_strategies.regime_thresholds_4h)
bull_strategy = BullTrendPullbackStrategy(enabled=True)
store = FeatureStore(
    consumers=[detector, bull_strategy],
    extra_requests=[
        FeatureRequest('ema', 9, '1h'),
        FeatureRequest('ema', 50, '1h'),
        FeatureRequest('ema', 200, '1h'),
        FeatureRequest('rvol', 20, '1h'),
    ],
)

def download_and_build_aligned_universe(start_ms: int = START_MS, end_ms: int | None = None):
    clock.sync(client.get_server_time_ms)
    collector = CandleCollector(client=client, db=db, clock=clock)
    for sym in SYMBOLS:
        times_1h = db.close_times(sym, '1h', since_ms=0)
        resume_1h = times_1h[-1] if times_1h else start_ms
        n_1h = collector.backfill_historical_range(sym, '1h', start_time_ms=resume_1h, end_time_ms=end_ms)
        times_4h = db.close_times(sym, '4h', since_ms=0)
        resume_4h = times_4h[-1] if times_4h else start_ms
        n_4h = collector.backfill_historical_range(sym, '4h', start_time_ms=resume_4h, end_time_ms=end_ms)
        print(f'[{sym}] SQLite WAL atualizado (+{n_1h} velas 1h, +{n_4h} velas 4h)')
    df_btc_1h = load_frame(db, 'BTCUSDT', '1h', limit=100_000)
    aligned = {}
    for sym in SYMBOLS:
        df_1h = load_frame(db, sym, '1h', limit=100_000)
        df_4h = load_frame(db, sym, '4h', limit=30_000)
        built = store.build(df_1h, df_4h)
        aligned[sym] = align_btc_context(built, df_btc_1h[[c for c in ('close_time_ms', 'close') if c in df_btc_1h.columns]])
    return aligned


In [ ]:
import numpy as np
from src.backtest.causal_simulator import CausalSingleSlotSimulator, CostScenario
from src.backtest.ablation_and_dsr import (
    split_temporal_vaults,
    compute_afml_sample_uniqueness,
    compute_deflated_sharpe_ratio,
    register_backtest_trial,
    evaluate_gate3_criteria,
)

REGISTRY_CSV = Path('logs/backtest_trials_registry.csv')

def run_ablation_and_gate3(aligned_frames, intracandle_1m_map=None):
    tcol = 'close_time_ms' if 'close_time_ms' in next(iter(aligned_frames.values())).columns else 'close_time'
    all_ts = sorted({int(t) for df in aligned_frames.values() for t in df[tcol].tolist()})
    bounds = split_temporal_vaults(all_ts, train_pct=0.60, val_pct=0.25)

    sim_base = CausalSingleSlotSimulator(cost_scenario=CostScenario.BASE)
    sim_pess = CausalSingleSlotSimulator(cost_scenario=CostScenario.PESSIMISTIC)

    trades_step_b = sim_base.run_multi_asset_simulation(
        aligned_frames, [bull_strategy], detector, ignore_regime_filter=True,
        start_time_ms=bounds.val_start_ms, end_time_ms=bounds.holdout_end_ms
    )
    trades_step_c_pure = sim_base.run_multi_asset_simulation(
        aligned_frames, [bull_strategy], detector, require_order_flow_filter=False,
        start_time_ms=bounds.val_start_ms, end_time_ms=bounds.holdout_end_ms
    )
    trades_step_c_pess = sim_pess.run_multi_asset_simulation(
        aligned_frames, [bull_strategy], detector, require_order_flow_filter=False,
        start_time_ms=bounds.val_start_ms, end_time_ms=bounds.holdout_end_ms
    )
    trades_step_c_of = sim_base.run_multi_asset_simulation(
        aligned_frames, [bull_strategy], detector, require_order_flow_filter=True,
        start_time_ms=bounds.val_start_ms, end_time_ms=bounds.holdout_end_ms
    )
    sim_be = CausalSingleSlotSimulator(cost_scenario=CostScenario.BASE, breakeven_enabled=True, breakeven_trigger_r=1.2)
    trades_step_e = sim_be.run_multi_asset_simulation(
        aligned_frames, [bull_strategy], detector, intracandle_1m_map=intracandle_1m_map,
        start_time_ms=bounds.val_start_ms, end_time_ms=bounds.holdout_end_ms
    )

    gate3_pure = evaluate_gate3_criteria(
        trades_base=trades_step_c_pure,
        trades_pessimistic=trades_step_c_pess,
        holdout_start_ms=bounds.holdout_start_ms,
        n_trials=2,
    )
    return {
        'bounds': bounds,
        'step_b': trades_step_b,
        'step_c_pure': trades_step_c_pure,
        'step_c_order_flow': trades_step_c_of,
        'step_e_breakeven_1m': trades_step_e,
        'gate3_report': gate3_pure,
    }


In [ ]:
# Execução automática no Google Colab ao clicar em 'Executar tudo'
if 'google.colab' in sys.modules:
    if 'aligned_frames' not in globals():
        aligned_frames = download_and_build_aligned_universe()
    results = run_ablation_and_gate3(aligned_frames)
    for step_key in ('step_b', 'step_c_pure', 'step_c_order_flow', 'step_e_breakeven_1m'):
        t_list = results[step_key]
        mean_r = float(np.mean([t.net_r for t in t_list])) if t_list else 0.0
        print(f'{step_key:20s} | n_trades={len(t_list):4d} | E[R]_net={mean_r:+.4f}R')
    print('\n=== RELATÓRIO GATE 3 (DEGRAU C) ===')
    print(results['gate3_report'])
